In [1]:
#Errors, Book, BorrowRecord
from datetime import date, timedelta


class LibraryError(Exception):
    """Base error."""

class BookUnavailableError(LibraryError):
    """Book pehle se issue hai."""

class BorrowLimitError(LibraryError):
    """Member ki limit poori."""

class PrivilegeError(LibraryError):
    """Tier ko ye ijazat nahi."""

class OutstandingFineError(LibraryError):
    """Fine bakaya hai."""


class Book:
    def __init__(self, isbn, title, author, is_reference=False):
        self.isbn = isbn
        self.title = title
        self.author = author
        self.is_reference = is_reference
        self.is_available = True

    def __str__(self):
        tag = " (Reference)" if self.is_reference else ""
        status = "Available" if self.is_available else "Issued"
        return f"{self.title} by {self.author}{tag} [{self.isbn}] - {status}"

    def __repr__(self):
        return f"Book(isbn={self.isbn!r}, title={self.title!r})"

    def __eq__(self, other):
        return isinstance(other, Book) and self.isbn == other.isbn

    def __hash__(self):
        return hash(self.isbn)


class BorrowRecord:
    def __init__(self, book, member, borrowed_on, due_on):
        self.book = book
        self.member = member
        self.borrowed_on = borrowed_on
        self.due_on = due_on
        self.returned_on = None
        self.fine = 0.0

    def days_overdue(self, on):
        end = self.returned_on or on
        return max((end - self.due_on).days, 0)

    def __str__(self):
        if self.returned_on:
            status = f"returned {self.returned_on:%d %b} (fine Rs. {self.fine:,.2f})"
        else:
            status = "ACTIVE"
        return (f"{self.book.title:<22} -> {self.member.name:<8} | "
                f"{self.borrowed_on:%d %b} to {self.due_on:%d %b} | {status}")

In [2]:
#Member tiers (inheritance + overriding)
class Member:
    tier = "Member"
    max_books = 1
    loan_days = 7
    fine_per_day = 10.0
    fine_block_limit = 0.0          # itne fine tak borrow allowed
    can_borrow_reference = False

    def __init__(self, member_id, name):
        self.member_id = member_id
        self.name = name
        self.loans = []             # active BorrowRecords
        self.fines_due = 0.0

    # ---------- fine logic (har tier override karta hai) ----------
    def calculate_fine(self, days_overdue):
        raise NotImplementedError("Tier ko calculate_fine override karna hoga")

    # ---------- borrowing rules ----------
    def check_can_borrow(self, book):
        if self.fines_due > self.fine_block_limit:
            raise OutstandingFineError(
                f"{self.name}: Rs. {self.fines_due:,.2f} fine bakaya hai "
                f"(limit Rs. {self.fine_block_limit:,.2f})")
        if book.is_reference and not self.can_borrow_reference:
            raise PrivilegeError(f"{self.tier} member reference book issue nahi kar sakta")
        if len(self) >= self.max_books:
            raise BorrowLimitError(
                f"{self.name}: {self.tier} limit {self.max_books} books hai")

    # ---------- dunder methods ----------
    def __len__(self):
        return len(self.loans)

    def __contains__(self, book):
        return any(r.book == book for r in self.loans)

    def __iter__(self):
        return (r.book for r in self.loans)

    def __eq__(self, other):
        return isinstance(other, Member) and self.member_id == other.member_id

    def __hash__(self):
        return hash(self.member_id)

    def __str__(self):
        return (f"[{self.tier:<8}] {self.member_id} {self.name:<8} | "
                f"loans {len(self)}/{self.max_books} | fines due Rs. {self.fines_due:,.2f}")

    def __repr__(self):
        return f"{type(self).__name__}({self.member_id!r}, {self.name!r})"


# ---------- Single inheritance ----------
class StandardMember(Member):
    tier = "Standard"
    max_books = 2
    loan_days = 14
    fine_per_day = 10.0
    fine_cap = 500.0

    def calculate_fine(self, days_overdue):
        return min(days_overdue * self.fine_per_day, self.fine_cap)


class PremiumMember(Member):
    tier = "Premium"
    max_books = 5
    loan_days = 21
    fine_per_day = 5.0
    grace_days = 3
    fine_block_limit = 50.0
    can_borrow_reference = True

    def calculate_fine(self, days_overdue):
        billable = max(days_overdue - self.grace_days, 0)
        return billable * self.fine_per_day


# ---------- Multilevel inheritance: Member -> PremiumMember -> VIPMember ----------
class VIPMember(PremiumMember):
    tier = "VIP"
    max_books = 10
    loan_days = 30
    grace_days = 7
    fine_block_limit = 200.0
    fine_cap = 100.0
    discount = 0.5

    def calculate_fine(self, days_overdue):
        base = super().calculate_fine(days_overdue)       # Premium ka fine (7 din grace ke saath)
        return min(base * self.discount, self.fine_cap)

In [3]:
#Library class
class Library:
    def __init__(self, name):
        self.name = name
        self._books = {}
        self._members = {}
        self.records = []

    def add_book(self, book):
        self._books[book.isbn] = book

    def add_member(self, member):
        self._members[member.member_id] = member

    def _member(self, member_id):
        if member_id not in self._members:
            raise LibraryError(f"Member '{member_id}' registered nahi")
        return self._members[member_id]

    def _book(self, isbn):
        if isbn not in self._books:
            raise LibraryError(f"Book '{isbn}' catalogue mein nahi")
        return self._books[isbn]

    def borrow(self, member_id, isbn, on):
        member, book = self._member(member_id), self._book(isbn)
        member.check_can_borrow(book)
        if not book.is_available:
            raise BookUnavailableError(f"'{book.title}' pehle se issue hai")
        record = BorrowRecord(book, member, on, on + timedelta(days=member.loan_days))
        book.is_available = False
        member.loans.append(record)
        self.records.append(record)
        return record

    def return_book(self, member_id, isbn, on):
        member, book = self._member(member_id), self._book(isbn)
        record = next((r for r in member.loans if r.book == book), None)
        if record is None:
            raise LibraryError(f"{member.name} ke paas '{book.title}' nahi hai")
        record.returned_on = on
        record.fine = member.calculate_fine(record.days_overdue(on))
        member.fines_due += record.fine
        member.loans.remove(record)
        book.is_available = True
        return record.fine

    def pay_fines(self, member_id):
        member = self._member(member_id)
        paid, member.fines_due = member.fines_due, 0.0
        return paid

    def overdue_report(self, on):
        rows = []
        for r in self.records:
            if r.returned_on is None and on > r.due_on:
                days = r.days_overdue(on)
                rows.append((r, days, r.member.calculate_fine(days)))
        return rows

    # ---------- dunder methods ----------
    def __len__(self):
        return len(self._books)

    def __iter__(self):
        return iter(self._books.values())

    def __contains__(self, isbn):
        return isbn in self._books

    def __getitem__(self, isbn):
        return self._book(isbn)

    def __str__(self):
        issued = sum(not b.is_available for b in self)
        line = "=" * 66
        out = [line, f"{self.name.upper()}".center(66), line,
               f"Books: {len(self)} (issued: {issued}) | Members: {len(self._members)}",
               "-" * 66, "MEMBERS"]
        out += [f"  {m}" for m in self._members.values()]
        out += ["-" * 66, "BOOKS"]
        out += [f"  {b}" for b in self]
        out.append(line)
        return "\n".join(out)

In [4]:
#Demo
lib = Library("City Library")
for b in [
    Book("B1", "Python Basics",     "Guido"),
    Book("B2", "Data Structures",   "Knuth"),
    Book("B3", "Machine Learning",  "Ng"),
    Book("B4", "Clean Code",        "Martin"),
    Book("B5", "Encyclopedia Vol.1", "Various", is_reference=True),
    Book("B6", "Deep Learning",     "Goodfellow"),
]:
    lib.add_book(b)

lib.add_member(StandardMember("M001", "Ali"))
lib.add_member(PremiumMember("M002", "Sara"))
lib.add_member(VIPMember("M003", "Malika"))


def attempt(label, fn, *args):
    try:
        result = fn(*args)
        print(f"✅ {label:<34} {result if result is not None else ''}")
    except LibraryError as e:
        print(f"❌ {label:<34} {type(e).__name__}: {e}")


d = lambda n: date(2026, 10, 1) + timedelta(days=n)     # 1 Oct 2026 se n din baad

print("--- 1 Oct: borrowing ---")
attempt("Ali borrows B1",               lib.borrow, "M001", "B1", d(0))
attempt("Ali borrows B2",               lib.borrow, "M001", "B2", d(0))
attempt("Ali borrows B3 (limit)",       lib.borrow, "M001", "B3", d(0))
attempt("Ali borrows B5 (reference)",   lib.borrow, "M001", "B5", d(0))
attempt("Sara borrows B3",              lib.borrow, "M002", "B3", d(0))
attempt("Sara borrows B1 (issued)",     lib.borrow, "M002", "B1", d(0))
attempt("Malika borrows B5 (reference)", lib.borrow, "M003", "B5", d(0))
attempt("Malika borrows B4",            lib.borrow, "M003", "B4", d(0))

print("\n--- Returns (alag tiers, alag fines) ---")
attempt("Ali returns B1 (25 Oct)",      lambda: f"fine Rs. {lib.return_book('M001', 'B1', d(24)):,.2f}")
attempt("Ali borrows B6 (fine bakaya)", lib.borrow, "M001", "B6", d(24))
attempt("Ali pays fines",               lambda: f"paid Rs. {lib.pay_fines('M001'):,.2f}")
attempt("Ali borrows B6 (25 Oct)",      lib.borrow, "M001", "B6", d(24))
attempt("Sara returns B3 (29 Oct)",     lambda: f"fine Rs. {lib.return_book('M002', 'B3', d(28)):,.2f}")
attempt("Malika returns B5 (15 Nov)",   lambda: f"fine Rs. {lib.return_book('M003', 'B5', d(45)):,.2f}")

print("\n--- Overdue report (15 Nov) ---")
for r, days, fine in lib.overdue_report(d(45)):
    print(f"  {r.member.name:<8} {r.book.title:<18} {days:>3} din overdue -> Rs. {fine:>7,.2f}")

print("\n--- Dunder methods ---")
ali = lib._member("M001")
print("len(ali) :", len(ali), "| B2 in ali:", lib["B2"] in ali, "| B1 in ali:", lib["B1"] in ali)
print("Ali ki books:", [b.title for b in ali])
print("repr     :", repr(ali), "|", repr(lib["B1"]))
print("VIP MRO  :", " -> ".join(c.__name__ for c in VIPMember.__mro__[:-1]))
print("isinstance(VIP, PremiumMember):", isinstance(lib._member("M003"), PremiumMember))

print("\n--- Borrowing history ---")
print("\n".join(str(r) for r in lib.records))

print("\n" + str(lib))

--- 1 Oct: borrowing ---
✅ Ali borrows B1                     Python Basics          -> Ali      | 01 Oct to 15 Oct | ACTIVE
✅ Ali borrows B2                     Data Structures        -> Ali      | 01 Oct to 15 Oct | ACTIVE
❌ Ali borrows B3 (limit)             BorrowLimitError: Ali: Standard limit 2 books hai
❌ Ali borrows B5 (reference)         PrivilegeError: Standard member reference book issue nahi kar sakta
✅ Sara borrows B3                    Machine Learning       -> Sara     | 01 Oct to 22 Oct | ACTIVE
❌ Sara borrows B1 (issued)           BookUnavailableError: 'Python Basics' pehle se issue hai
✅ Malika borrows B5 (reference)      Encyclopedia Vol.1     -> Malika   | 01 Oct to 31 Oct | ACTIVE
✅ Malika borrows B4                  Clean Code             -> Malika   | 01 Oct to 31 Oct | ACTIVE

--- Returns (alag tiers, alag fines) ---
✅ Ali returns B1 (25 Oct)            fine Rs. 100.00
❌ Ali borrows B6 (fine bakaya)       OutstandingFineError: Ali: Rs. 100.00 fine bakaya hai (li